# Topic 47 — Kazakhstan OpenStreetMap POI Analytics + GPU Model Training

**One-click Colab workflow:** downloads the real Kazakhstan PBF directly from [Geofabrik](https://download.geofabrik.de/asia/kazakhstan.html), extracts OSM points of interest, **trains your own PyTorch GPU autoencoder** to learn neighbourhood amenity profiles, evaluates it against two baselines, and runs the mandatory Topic 47 analytics (regional comparisons, geographic DBSCAN, SQLite, EDA and Pandas/Polars benchmarks).

## Important distinction
- **Trainable AI model:** a denoising/reconstruction **autoencoder** learns patterns in each 2 km grid cell's POI-category counts. It is trained on the GPU using a train/validation/test spatial split. This is *unsupervised neural-network training* on real OSM data, not classification with labeled ground truth.
- **DBSCAN:** separately finds **geographic concentrations of POIs** using haversine distance, as required by the topic. DBSCAN is a clustering algorithm and **does not train a neural network**.
- **GPU accelerates the autoencoder training;** PBF extraction, spatial joins, SQL and scikit-learn DBSCAN remain CPU tasks.

### How to run
1. Upload this `.ipynb` to Google Colab.
2. Select **Runtime → Change runtime type → T4 GPU** (or any available GPU).
3. Click **Runtime → Run all**. No dataset upload is required; the notebook downloads it itself. Requires enough free disk space for a ~200–250 MB PBF plus intermediates and Internet access.
4. Download `topic47_app_data.zip` from the last cell to hand to Codex.

**Sources:** © OpenStreetMap contributors (ODbL), Geofabrik download; Kazakhstan Bureau of National Statistics (population for 2026-10-01). Data completeness and POI coverage vary by area. All analytical results are computed at run time.


## 0 · Requirements checklist

- Large dataset extraction with memory-conscious streaming and Parquet, type/memory audit.
- Baseline **SQLite GROUP BY**, plus **Pandas** and **Polars** group-by on the *same underlying rows*; wall time, peak process RSS, result equality.
- EDA, outliers/missing data, two explicit hypotheses, five+ visuals and interactive Plotly map.
- 2026 population per 100k, area-normalized POI counts, Astana vs Almaty vs Shymkent, DBSCAN in geographic space.
- Normalized SQLite tables, ER diagram, **8 SQL analytical queries**, query exports back to Pandas.
- Outputs under `outputs/` and a clear contract for Streamlit/Codex.

**Interpretation rule:** OSM measures *mapped features*, not a census of all real amenities. Points, ways and relations represent heterogeneous OSM geometries; representative points of polygons are approximations for display and proximity analyses. Labels and per-capita rates are descriptive, not direct measurements of service accessibility.
- **Trained GPU model extension:** PyTorch autoencoder on log-POI-count grid vectors, group-disjoint spatial validation/test split, PCA/constant baselines, saved checkpoint + scaler + evaluation report; geographic DBSCAN is retained separately.


## 1 · Colab setup

**Before running:** `Runtime → Change runtime type → T4 GPU`. This notebook checks CUDA before downloading the dataset, so a CPU runtime cannot silently run a supposed GPU experiment. `torch` is already installed in current Colab GPU runtimes; the other dependencies and the `osmium` command-line utility are installed below.


In [1]:
import importlib.util, os, shutil, subprocess, sys

PIP_PACKAGES = {
    "pandas": "pandas>=2.1,<3", "numpy": "numpy>=1.26,<3",
    "pyarrow": "pyarrow>=16,<25", "polars": "polars>=1.10,<3",
    "geopandas": "geopandas>=1.0,<2", "shapely": "shapely>=2.0,<3",
    "pyproj": "pyproj>=3.6,<4", "plotly": "plotly>=5.20,<7",
    "scipy": "scipy>=1.11,<2", "sklearn": "scikit-learn>=1.4,<2",
    "psutil": "psutil>=5.9,<9", "requests": "requests>=2.31,<3",
    "folium": "folium>=0.16,<1", "torch": "torch>=2.2,<3",
}
missing = [pkg for module, pkg in PIP_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if not shutil.which("osmium"):
    if sys.platform.startswith("linux") and hasattr(os, "geteuid") and os.geteuid() == 0:
        subprocess.check_call(["apt-get", "update", "-qq"])
        subprocess.check_call(["apt-get", "install", "-y", "-qq", "osmium-tool"])
    else:
        raise RuntimeError("Install osmium-tool (Ubuntu: sudo apt install osmium-tool) and rerun this cell")
print(subprocess.check_output(["osmium", "--version"], text=True).splitlines()[0])

osmium version 1.16.0


In [2]:
# Confirm that Colab is actually using a GPU BEFORE the PBF download.
import torch
REQUIRE_CUDA = True
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. In Colab select Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)
if DEVICE.type == "cuda":
    print("NVIDIA GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


Training device: cuda
NVIDIA GPU: Tesla T4
PyTorch: 2.14.1+cu130


In [3]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, hashlib, json, math, re, sqlite3, tempfile, time, unicodedata
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import geopandas as gpd
from shapely.geometry import shape, Point
from shapely import wkt
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.cluster import DBSCAN
from sklearn.neighbors import BallTree
import plotly.express as px
import plotly.graph_objects as go
import requests

ROOT = Path.cwd() / "kazakhstan_poi_47"
RAW = ROOT / "data" / "raw"
OUT = ROOT / "outputs"
FIG = OUT / "figures"
QUERY_DIR = OUT / "sql_results"
for p in (RAW, OUT, FIG, QUERY_DIR): p.mkdir(parents=True, exist_ok=True)
PBF = RAW / "kazakhstan-latest.osm.pbf"
POI_PBF = RAW / "poi-filtered.osm.pbf"
REGION_PBF = RAW / "adm1-filtered.osm.pbf"
POI_SEQ = RAW / "poi.geojsonseq"
REGION_SEQ = RAW / "adm1.geojsonseq"
RAW_PQ = OUT / "poi_raw.parquet"
ENRICHED_PQ = OUT / "poi_enriched.parquet"
REGION_GJ = OUT / "regions.geojson"
REGION_CSV = OUT / "regions.csv"
DB_PATH = OUT / "kazakhstan_poi.sqlite"
PBF_URL = "https://download.geofabrik.de/asia/kazakhstan-latest.osm.pbf"
DOWNLOAD_IF_MISSING = True
CHUNK_ROWS = 20000
MAX_CLUSTER_POINTS = 12000
RANDOM_STATE = 47
print("Project root:", ROOT)

Project root: /content/kazakhstan_poi_47


## 2 · Download the official Geofabrik PBF

Uses a streamed HTTP download with retries and a `.part` file. Full PBF is **not** checked into GitHub. Retain source URL + retrieval time in the final manifest; repeated download of `latest` may change numbers.

In [4]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def download_stream(url: str, dest: Path, chunk_size=1024 * 1024):
    dest.parent.mkdir(parents=True, exist_ok=True)
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=Retry(total=4, backoff_factor=2, status_forcelist=[429,500,502,503,504])))
    temp_path = dest.with_name(dest.name + ".part")
    with session.get(url, stream=True, timeout=(15, 180)) as response:
        response.raise_for_status()
        with temp_path.open("wb") as output:
            for block in response.iter_content(chunk_size=chunk_size):
                if block: output.write(block)
    temp_path.replace(dest)
    return dest

if not PBF.exists():
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError(f"Put a Kazakhstan PBF at {PBF}")
    download_stream(PBF_URL, PBF)
assert PBF.stat().st_size > 100_000_000, "Expected full Kazakhstan extract; check your file"
print(f"PBF: {PBF.name} ({PBF.stat().st_size / 1024**2:,.1f} MiB)")
PBF_LOCAL_FILE_MTIME_UTC = datetime.fromtimestamp(PBF.stat().st_mtime, timezone.utc).isoformat()

PBF: kazakhstan-latest.osm.pbf (215.8 MiB)


## 3 · OSM tag filtering and geometry export

Two separate `osmium tags-filter` passes select (A) POIs from `amenity`, `shop`, `tourism`, `leisure`, and (B) administrative relations tagged `admin_level=4`. In the second stage `osmium export` assembles ways and multipolygon relations and produces **newline-delimited GeoJSON**. This is essential because coordinates for ways/relations are not stored as a single node. `osmium` might omit invalid geometries; the quality report acknowledges this. The `-t` option clears tags on merely referenced members, avoiding spurious POIs.

**Administrative safeguard:** we later require all 20 present-day Kazakhstan regions/cities to be matched to the official population list. If some are missing, fix OSM name aliases; *do not* silently switch to old boundaries.

In [5]:
EXPORT_CONF = RAW / "osmium-export-config.json"
EXPORT_CONF.write_text(json.dumps({
    "attributes": {"type": True, "id": True, "version": False, "timestamp": False,
                   "changeset": False, "uid": False, "user": False, "way_nodes": False},
    "linear_tags": ["highway", "waterway", "railway", "barrier"],
    "area_tags": ["amenity", "shop", "tourism", "leisure", "boundary"],
    "include_tags": ["amenity", "shop", "tourism", "leisure", "name", "name:en", "name:ru",
                     "name:kk", "boundary", "admin_level", "type", "area"],
    "format_options": {"print_record_separator": False}
}, indent=2), encoding="utf-8")

def run_cmd(args):
    print("$", " ".join(map(str, args)))
    subprocess.run([str(x) for x in args], check=True)

if not POI_SEQ.exists():
    if not POI_PBF.exists():
        run_cmd(["osmium", "tags-filter", "-t", PBF,
                 "nwr/amenity", "nwr/shop", "nwr/tourism", "nwr/leisure", "-o", POI_PBF])
    run_cmd(["osmium", "export", POI_PBF, "-o", POI_SEQ, "-f", "geojsonseq", "-c", EXPORT_CONF])
if not REGION_SEQ.exists():
    if not REGION_PBF.exists():
        run_cmd(["osmium", "tags-filter", "-t", PBF, "r/admin_level=4", "-o", REGION_PBF])
    run_cmd(["osmium", "export", REGION_PBF, "-o", REGION_SEQ,
             "-f", "geojsonseq", "--geometry-types=polygon", "-c", EXPORT_CONF])
print("Streaming sequence files:", POI_SEQ.stat().st_size, REGION_SEQ.stat().st_size)

$ osmium tags-filter -t /content/kazakhstan_poi_47/data/raw/kazakhstan-latest.osm.pbf nwr/amenity nwr/shop nwr/tourism nwr/leisure -o /content/kazakhstan_poi_47/data/raw/poi-filtered.osm.pbf
$ osmium export /content/kazakhstan_poi_47/data/raw/poi-filtered.osm.pbf -o /content/kazakhstan_poi_47/data/raw/poi.geojsonseq -f geojsonseq -c /content/kazakhstan_poi_47/data/raw/osmium-export-config.json
$ osmium tags-filter -t /content/kazakhstan_poi_47/data/raw/kazakhstan-latest.osm.pbf r/admin_level=4 -o /content/kazakhstan_poi_47/data/raw/adm1-filtered.osm.pbf
$ osmium export /content/kazakhstan_poi_47/data/raw/adm1-filtered.osm.pbf -o /content/kazakhstan_poi_47/data/raw/adm1.geojsonseq -f geojsonseq --geometry-types=polygon -c /content/kazakhstan_poi_47/data/raw/osmium-export-config.json
Streaming sequence files: 33123919 2641208


## 4 · Stream POIs into partition-friendly Parquet

The *main* analytics definition is **one row per OSM entity** (node, way or relation) with a prioritized main tag; entities with multiple POI tags are not double-counted. For lines/polygons we store a **representative point** for maps/clustering. OSM IDs are reused across object types, so the primary key includes `node/way/relation`. Duplicate geometric representations of the same OSM way are dropped with a stable key. Unknown amenity values remain in `Other amenities`, not discarded.

In [6]:
CATEGORY_GROUPS = {
    "Food & drink": {"restaurant", "cafe", "fast_food", "bar", "pub", "food_court", "ice_cream", "biergarten"},
    "Healthcare": {"hospital", "clinic", "doctors", "dentist", "pharmacy", "veterinary", "health_post"},
    "Education": {"school", "kindergarten", "college", "university", "library", "language_school", "music_school"},
    "Finance": {"bank", "atm", "bureau_de_change", "money_transfer", "payment_terminal"},
    "Transport": {"bus_station", "fuel", "parking", "bicycle_parking", "charging_station", "taxi", "car_sharing"},
    "Public services": {"police", "fire_station", "post_office", "townhall", "courthouse", "public_building"},
}
KEY_TO_GROUP = {value: group for group, values in CATEGORY_GROUPS.items() for value in values}
POI_KEYS = ("amenity", "shop", "tourism", "leisure")
RAW_SCHEMA = pa.schema([
    ("osm_key", pa.string()), ("osm_type", pa.string()), ("osm_id", pa.int64()),
    ("name", pa.string()), ("main_tag", pa.string()), ("subcategory", pa.string()),
    ("category", pa.string()), ("latitude", pa.float64()), ("longitude", pa.float64()),
    ("geometry_type", pa.string())
])

def geojson_records(path):
    with Path(path).open(encoding="utf-8") as fh:
        for line in fh:
            line = line.lstrip("\x1e").strip()
            if line:
                try: yield json.loads(line)
                except json.JSONDecodeError: continue

def classify_poi(props):
    for key in POI_KEYS:
        value = str(props.get(key) or "").strip().lower()
        if not value or value in {"no", "none", "false", "disused"}: continue
        if key == "shop": category = "Retail"
        elif key == "tourism": category = "Tourism"
        elif key == "leisure": category = "Recreation"
        else: category = KEY_TO_GROUP.get(value, "Other amenities")
        return key, value, category
    return None

def poi_from_feature(feature):
    props = feature.get("properties") or {}
    tag = classify_poi(props)
    if tag is None: return None
    osm_type, osm_id = props.get("@type"), props.get("@id")
    if osm_type not in {"node", "way", "relation"} or osm_id is None: return None
    geometry = feature.get("geometry")
    if not geometry: return None
    try:
        geom = shape(geometry)
        if geom.is_empty: return None
        point = geom if isinstance(geom, Point) else geom.representative_point()
        longitude, latitude = float(point.x), float(point.y)
        if not (46 <= longitude <= 88 and 40 <= latitude <= 56): return None
        osm_id = int(osm_id)
    except (ValueError, TypeError, KeyError): return None
    main_tag, subcategory, category = tag
    return {"osm_key": f"{osm_type}:{osm_id}", "osm_type": osm_type, "osm_id": osm_id,
            "name": (props.get("name:en") or props.get("name") or None),
            "main_tag": main_tag, "subcategory": subcategory, "category": category,
            "latitude": latitude, "longitude": longitude, "geometry_type": geom.geom_type}

print("Example parser:", poi_from_feature({
    "type": "Feature", "properties": {"@type": "node", "@id": 123, "amenity": "restaurant", "name": "Demo"},
    "geometry": {"type": "Point", "coordinates": [71.45, 51.15]}}))

Example parser: {'osm_key': 'node:123', 'osm_type': 'node', 'osm_id': 123, 'name': 'Demo', 'main_tag': 'amenity', 'subcategory': 'restaurant', 'category': 'Food & drink', 'latitude': 51.15, 'longitude': 71.45, 'geometry_type': 'Point'}


In [7]:
def stream_pois_to_parquet(source, destination, batch_rows=20000):
    writer = pq.ParquetWriter(destination, RAW_SCHEMA, compression="zstd")
    row_buffer = []
    total_features = 0
    dropped = 0
    seen = set()  # one compact entity key per POI (needed for area/line duplication)
    try:
        for feature in geojson_records(source):
            total_features += 1
            row = poi_from_feature(feature)
            if row is None or row["osm_key"] in seen:
                dropped += 1
                continue
            seen.add(row["osm_key"])
            row_buffer.append(row)
            if len(row_buffer) >= batch_rows:
                writer.write_table(pa.Table.from_pylist(row_buffer, schema=RAW_SCHEMA))
                row_buffer.clear()
        if row_buffer: writer.write_table(pa.Table.from_pylist(row_buffer, schema=RAW_SCHEMA))
    finally:
        writer.close()
    return {"geojson_features": total_features, "poi_records": len(seen), "skipped_or_duplicates": dropped}

if not RAW_PQ.exists():
    POI_EXTRACT_STATS = stream_pois_to_parquet(POI_SEQ, RAW_PQ, CHUNK_ROWS)
else:
    POI_EXTRACT_STATS = {"poi_records": pq.read_metadata(RAW_PQ).num_rows, "cached": True}
print(POI_EXTRACT_STATS)
assert POI_EXTRACT_STATS["poi_records"] > 0, "No POIs extracted — inspect the osmium export"

{'geojson_features': 128992, 'poi_records': 126778, 'skipped_or_duplicates': 2214}


## GPU extension · Train your own AI model on Kazakhstan OSM data

**Task:** represent each 2 km grid cell by the numbers of Food & drink, Healthcare, Education, Retail and other OSM POI categories. Learn a compact **4-dimensional latent representation** using a PyTorch autoencoder. This lets you compare neighbourhood amenity profiles and discover types of urban areas.

**Training signal:** reconstruct the original log-transformed category-count vector. There are no externally provided labels. **Train/validation/test are disjoint by 10 km spatial blocks**, avoiding random splitting of neighbouring 2 km cells.

**Baselines:** (a) always reconstruct the training mean, (b) PCA reconstruction with the same number of latent dimensions. Compare held-out MSE and MAE; a neural network **is not guaranteed** to outperform PCA. All model artifacts are saved in `outputs/`.

**Scope:** only grid cells with at least `MIN_POIS_PER_CELL` mapped POIs are modeled. The cell counts reflect *OSM mapping coverage*, not real-world amenity completeness. Geographic DBSCAN below is a **different analysis**.


In [8]:
from pyproj import Transformer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib, random

GRID_SIZE_M = 2000                 # 2 km amenity-composition cells
SPLIT_BLOCK_M = 10000             # disjoint 10 km geographic blocks
MIN_POIS_PER_CELL = 3
LATENT_DIM_REQUESTED = 4
SEED = 47

raw_cols = pd.read_parquet(RAW_PQ, columns=["longitude", "latitude", "category"])
raw_cols = raw_cols.dropna(subset=["longitude", "latitude", "category"])
assert len(raw_cols) > 0, "No POIs available from raw Parquet"

forward = Transformer.from_crs("EPSG:4326", "EPSG:6933", always_xy=True)
backward = Transformer.from_crs("EPSG:6933", "EPSG:4326", always_xy=True)
xs, ys = forward.transform(raw_cols.longitude.to_numpy(), raw_cols.latitude.to_numpy())
raw_cols["grid_x"] = np.floor(xs / GRID_SIZE_M).astype("int32")
raw_cols["grid_y"] = np.floor(ys / GRID_SIZE_M).astype("int32")

# Count every POI category within each grid cell. No labels are invented.
category_names = sorted(raw_cols.category.unique().tolist())
grid_counts = (raw_cols.groupby(["grid_x", "grid_y", "category"], observed=True)
               .size().unstack(fill_value=0).reindex(columns=category_names, fill_value=0))
grid_counts["total_pois"] = grid_counts.sum(axis=1)
grid_counts = grid_counts.loc[grid_counts.total_pois >= MIN_POIS_PER_CELL].reset_index()
assert len(grid_counts) >= 100, (f"Only {len(grid_counts)} cells meet MIN_POIS_PER_CELL; "
                                 "try setting MIN_POIS_PER_CELL=1 and re-running this cell")
assert len(category_names) >= 2, "At least two POI categories are needed"
latent_dim = min(LATENT_DIM_REQUESTED, len(category_names)-1)

# Spatial block groups are shared by all cells in the same 10 km square.
block_factor = SPLIT_BLOCK_M // GRID_SIZE_M
assert SPLIT_BLOCK_M % GRID_SIZE_M == 0
xblock = np.floor_divide(grid_counts.grid_x.to_numpy(), block_factor)
yblock = np.floor_divide(grid_counts.grid_y.to_numpy(), block_factor)
spatial_groups = np.array([f"{a}:{b}" for a,b in zip(xblock,yblock)])
assert len(set(spatial_groups)) >= 15, "Not enough spatially independent blocks for splits"

X_raw = np.log1p(grid_counts[category_names].to_numpy(dtype=np.float32))
trainval_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
                              .split(X_raw, groups=spatial_groups))
tr_sub, val_sub = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED+1)
                       .split(X_raw[trainval_idx], groups=spatial_groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr_sub], trainval_idx[val_sub]
assert not (set(spatial_groups[train_idx]) & set(spatial_groups[val_idx]))
assert not (set(spatial_groups[train_idx]) & set(spatial_groups[test_idx]))
assert not (set(spatial_groups[val_idx]) & set(spatial_groups[test_idx]))
scaler = StandardScaler().fit(X_raw[train_idx])  # fit on train ONLY
X = scaler.transform(X_raw).astype("float32")

print("OSM POI rows:", f"{len(raw_cols):,}")
print("Amenity category features:", category_names)
print("Grid cells:", f"{len(grid_counts):,}", "| Training/Validation/Test:",
      len(train_idx), len(val_idx), len(test_idx))
print("Spatial groups:", len(set(spatial_groups)), "| Device:", DEVICE)


OSM POI rows: 126,778
Amenity category features: ['Education', 'Finance', 'Food & drink', 'Healthcare', 'Other amenities', 'Public services', 'Recreation', 'Retail', 'Tourism', 'Transport']
Grid cells: 3,805 | Training/Validation/Test: 2478 606 721
Spatial groups: 1498 | Device: cuda


### Train the model on the GPU

The model is an original small fully connected autoencoder trained with Adam. Early stopping selects the best validation epoch; **the test blocks are not used for model selection**.

In [9]:
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

class AmenityAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(),
                                     nn.Linear(64, 32), nn.ReLU(),
                                     nn.Linear(32, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(),
                                     nn.Linear(32, 64), nn.ReLU(),
                                     nn.Linear(64, input_dim))

    def forward(self, x):
        return self.decoder(self.encoder(x))

EPOCHS = 70
BATCH_SIZE = 512
PATIENCE = 10
LR = 1e-3

train_loader = DataLoader(TensorDataset(torch.from_numpy(X[train_idx])), batch_size=BATCH_SIZE,
                          shuffle=True, pin_memory=(DEVICE.type == "cuda"))
val_tensor = torch.from_numpy(X[val_idx]).to(DEVICE)
model = AmenityAutoencoder(X.shape[1], latent_dim).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
loss_fn = nn.MSELoss()
best_val = float("inf")
best_state = None
no_improvement = 0
train_curve, val_curve = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for (batch_cpu,) in train_loader:
        batch = batch_cpu.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(batch)
        loss = loss_fn(prediction, batch)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(batch)
    train_mse = running / len(train_idx)

    model.eval()
    with torch.no_grad():
        # Exact held-out validation MSE; the validation groups never participate in optimizer updates.
        vpred = model(val_tensor)
        val_mse = loss_fn(vpred, val_tensor).item()
    train_curve.append(train_mse)
    val_curve.append(val_mse)

    if val_mse < best_val - 1e-5:
        best_val = val_mse
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        no_improvement = 0
    else:
        no_improvement += 1
    if epoch == 1 or epoch % 10 == 0 or no_improvement >= PATIENCE:
        print(f"Epoch {epoch:02d} | train MSE {train_mse:.5f} | val MSE {val_mse:.5f} | device={DEVICE}")
    if no_improvement >= PATIENCE:
        print(f"Early stopped at epoch {epoch}; best validation epoch {best_epoch}")
        break

assert best_state is not None
model.load_state_dict(best_state)
model.eval()
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))


Epoch 01 | train MSE 1.00866 | val MSE 0.74726 | device=cuda
Epoch 10 | train MSE 0.41296 | val MSE 0.41705 | device=cuda
Epoch 20 | train MSE 0.25454 | val MSE 0.24064 | device=cuda
Epoch 30 | train MSE 0.19543 | val MSE 0.18279 | device=cuda
Epoch 40 | train MSE 0.15972 | val MSE 0.15081 | device=cuda
Epoch 50 | train MSE 0.14610 | val MSE 0.14002 | device=cuda
Epoch 60 | train MSE 0.13971 | val MSE 0.13454 | device=cuda
Epoch 70 | train MSE 0.13127 | val MSE 0.12640 | device=cuda
Trainable parameters: 5838


### Evaluate on unseen spatial blocks and export the trained model

The comparison uses two reconstruction baselines on the **identical** held-out set. Lower MSE/MAE is better. The validation set drives early stopping; the test set is used once for the final comparison.

In [10]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from datetime import datetime, timezone

# Untouched test set: compare a constant-training-mean baseline, PCA, and GPU-trained autoencoder.
Xtest = X[test_idx]
with torch.no_grad():
    pred = model(torch.from_numpy(Xtest).to(DEVICE)).cpu().numpy()

pca = PCA(n_components=latent_dim, random_state=SEED).fit(X[train_idx])
pca_pred = pca.inverse_transform(pca.transform(Xtest))
mean_pred = np.zeros_like(Xtest)  # standardized train-set mean = zero

def calculate_errors(truth, estimate):
    return {"mse": float(np.mean((truth-estimate)**2)),
            "mae": float(np.mean(np.abs(truth-estimate)))}

metrics = {
    "train_mean_baseline": calculate_errors(Xtest, mean_pred),
    "pca_baseline": calculate_errors(Xtest, pca_pred),
    "gpu_autoencoder": calculate_errors(Xtest, pred),
}
print("Unseen spatial-block test metrics (standardized log-count space):")
print(pd.DataFrame(metrics).T.round(5).to_string())

# Create 4D embeddings for all qualifying cells; use MiniBatch-friendly batched GPU inference.
encoded_parts = []
with torch.no_grad():
    for start in range(0, len(X), 4096):
        encoded_parts.append(model.encoder(torch.from_numpy(X[start:start+4096]).to(DEVICE)).cpu().numpy())
embeddings = np.concatenate(encoded_parts, axis=0)
num_profiles = min(4, len(embeddings))
profile_kmeans = KMeans(n_clusters=num_profiles, n_init=10, random_state=SEED).fit(embeddings)
profile_labels = profile_kmeans.labels_.astype("int16")

# Grid cell centres projected back to WGS84 for dashboard mapping.
cent_x = (grid_counts.grid_x.to_numpy() + 0.5) * GRID_SIZE_M
cent_y = (grid_counts.grid_y.to_numpy() + 0.5) * GRID_SIZE_M
lon_cent, lat_cent = backward.transform(cent_x, cent_y)
profiles = grid_counts[["grid_x", "grid_y", "total_pois"]].copy()
profiles["longitude"] = lon_cent
profiles["latitude"] = lat_cent
for j, name in enumerate(category_names):
    profiles[f"category_count_{j:02d}"] = grid_counts[name].astype("int32")
for j in range(latent_dim):
    profiles[f"embedding_{j:02d}"] = embeddings[:,j]
profiles["profile_cluster"] = profile_labels
profiles.to_parquet(OUT / "urban_grid_profiles.parquet", index=False)

# Portable model, fitted preprocessor, and *unseen test* results for your future app and report.
model_package = {
    "state_dict": best_state, "input_dim": int(X.shape[1]),
    "latent_dim": int(latent_dim), "categories": category_names,
    "grid_size_m": GRID_SIZE_M,
    "feature_transform": "log1p(count) -> StandardScaler(train-only)",
}
torch.save(model_package, OUT / "urban_autoencoder.pt")
joblib.dump(scaler, OUT / "urban_autoencoder_scaler.joblib")
joblib.dump(pca, OUT / "pca_baseline.joblib")
joblib.dump(profile_kmeans, OUT / "urban_profile_kmeans.joblib")

report_ml = {
    "model_name": "AmenityAutoencoder",
    "task": "unsupervised neighbourhood POI-category profile reconstruction",
    "trained_device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else None,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "features": category_names,
    "category_count_columns": {f"category_count_{j:02d}":name for j,name in enumerate(category_names)},
    "grid_size_m": GRID_SIZE_M, "min_pois_per_cell": MIN_POIS_PER_CELL,
    "spatial_block_m": SPLIT_BLOCK_M,
    "train_cells": int(len(train_idx)), "validation_cells": int(len(val_idx)),
    "test_cells": int(len(test_idx)), "all_cells": int(len(grid_counts)),
    "training_epochs": int(len(train_curve)), "best_validation_epoch": int(best_epoch),
    "best_validation_mse": float(best_val),
    "parameter_count": int(sum(p.numel() for p in model.parameters())),
    "latent_dim": int(latent_dim), "test_metrics": metrics,
    "notes": ["Target equals input: unsupervised, not supervised amenity prediction",
              "Test groups disjoint from training and validation by 10 km cells",
              "MSE/MAE evaluated in standardized log-count feature space",
              "Profile KMeans labels are not geographic DBSCAN cluster IDs"],
}
(OUT / "gpu_model_report.json").write_text(json.dumps(report_ml, indent=2), encoding="utf-8")
pd.DataFrame(metrics).T.rename_axis("method").reset_index().to_csv(OUT / "gpu_model_metrics.csv", index=False)

fig_gpu = go.Figure()
fig_gpu.add_trace(go.Scatter(x=list(range(1, len(train_curve)+1)), y=train_curve, name="Train MSE"))
fig_gpu.add_trace(go.Scatter(x=list(range(1, len(val_curve)+1)), y=val_curve, name="Validation MSE"))
fig_gpu.update_layout(title="GPU autoencoder training history", xaxis_title="Epoch", yaxis_title="MSE")
fig_gpu.write_html(FIG / "00_gpu_autoencoder_training.html", include_plotlyjs="cdn")
fig_gpu.show()

fig_baseline = px.bar(pd.DataFrame(metrics).T.reset_index(names="approach"),
                      x="approach", y="mse", title="Held-out reconstruction error: baseline vs PCA vs GPU autoencoder")
fig_baseline.write_html(FIG / "00_gpu_model_comparison.html", include_plotlyjs="cdn")
fig_baseline.show()
print("Saved checkpoint, scaler, PCA baseline, profile clusters and reports to", OUT)


Unseen spatial-block test metrics (standardized log-count space):
                         mse      mae
train_mean_baseline  1.08332  0.79297
pca_baseline         0.17093  0.30338
gpu_autoencoder      0.12963  0.24350


Saved checkpoint, scaler, PCA baseline, profile clusters and reports to /content/kazakhstan_poi_47/outputs


### Model outputs / handoff for Codex

- `outputs/urban_autoencoder.pt`: **trained PyTorch model** and architecture metadata (`state_dict`, input dimension, categories, latent dimension).
- `outputs/urban_autoencoder_scaler.joblib`: preprocessing fit on training subset only.
- `outputs/pca_baseline.joblib`: fitted PCA comparison.
- `outputs/urban_profile_kmeans.joblib`: clustering algorithm on learned **profile embeddings** (not geographic density clusters).
- `outputs/gpu_model_report.json`, `gpu_model_metrics.csv`: spatial hold-out evaluation, GPU details and hyperparameters.
- `outputs/urban_grid_profiles.parquet`: grid cell coordinates, category counts, latent embeddings and profile cluster labels.
- `outputs/figures/00_gpu_autoencoder_training.html`, `00_gpu_model_comparison.html`: training/benchmark figures.

For a UI, **load the precomputed `urban_grid_profiles.parquet` first**. Running the trained autoencoder on new area data is an optional feature; if implemented, reconstruct exactly the original category order, `log1p` transform and saved `StandardScaler` before inference. Never present profile-cluster IDs as geographic DBSCAN IDs.


In [11]:
# Save the GPU-trained model and its app-ready feature data *before* the optional regional steps.
# This archive remains available even if an OSM boundary later needs manual name mapping.
import zipfile
EARLY_MODEL_ARCHIVE = ROOT / "topic47_gpu_model_outputs.zip"
files_to_package = [
    RAW_PQ, OUT / "urban_autoencoder.pt", OUT / "urban_autoencoder_scaler.joblib",
    OUT / "pca_baseline.joblib", OUT / "urban_profile_kmeans.joblib",
    OUT / "urban_grid_profiles.parquet", OUT / "gpu_model_report.json",
    OUT / "gpu_model_metrics.csv", FIG / "00_gpu_autoencoder_training.html",
    FIG / "00_gpu_model_comparison.html",
]
with zipfile.ZipFile(EARLY_MODEL_ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_package:
        assert f.exists(), f"Missing expected model output: {f}"
        zf.write(f, arcname=f.relative_to(ROOT))
print("GPU training archive:", EARLY_MODEL_ARCHIVE, "| MiB:", round(EARLY_MODEL_ARCHIVE.stat().st_size / 2**20, 1))
# For a browser download, uncomment and run:
# from google.colab import files
# files.download(str(EARLY_MODEL_ARCHIVE))

GPU training archive: /content/kazakhstan_poi_47/topic47_gpu_model_outputs.zip | MiB: 4.1


## 5 · Official regional populations (BNS, 1 October 2026)

These **20 official regions/cities of republican significance** total 20,633,529 people. They are kept as an explicit snapshot with source/date rather than guessed from OSM `population=*` tags. Source: <https://stat.gov.kz/en/region/> (October 1, 2026 view). Do not use total region population as if it were a city-only denominator for non-republic cities.

In [12]:
POPULATION_SOURCE = "https://stat.gov.kz/en/region/"
POPULATION_DATE = "2026-10-01"
POPULATION_ROWS = [
    ("astana", "Astana city", 1707150), ("almaty_city", "Almaty city", 2392105),
    ("shymkent", "Shymkent city", 1320743), ("almaty_region", "Almaty Region", 1618422),
    ("aktobe", "Aktobe Region", 959184), ("atyrau", "Atyrau Region", 718579),
    ("akmola", "Akmola Region", 796245), ("west_kazakhstan", "West Kazakhstan Region", 694278),
    ("east_kazakhstan", "East Kazakhstan Region", 714961), ("zhambyl", "Zhambyl Region", 1209574),
    ("karaganda", "Karaganda Region", 1130336), ("kostanay", "Kostanay Region", 818108),
    ("kyzylorda", "Kyzylorda Region", 844183), ("mangystau", "Mangystau Region", 827102),
    ("north_kazakhstan", "North Kazakhstan Region", 510181), ("pavlodar", "Pavlodar Region", 741603),
    ("turkistan", "Turkistan Region", 2141409), ("abay", "Abay Region", 590160),
    ("zhetisu", "Zhetisu Region", 682160), ("ulytau", "Ulytau Region", 217046),
]
population = pd.DataFrame(POPULATION_ROWS, columns=["region_code", "region_name", "population"])
population["population_date"] = POPULATION_DATE
assert len(population) == 20 and population.population.sum() == 20633529
population.to_csv(OUT / "population_2026_10_01.csv", index=False)
print("Official population total:", f"{population.population.sum():,}")
population.head()

Official population total: 20,633,529


,region_code,region_name,population,population_date
0,astana,Astana city,1707150,2026-10-01
1,almaty_city,Almaty city,2392105,2026-10-01
2,shymkent,Shymkent city,1320743,2026-10-01
3,almaty_region,Almaty Region,1618422,2026-10-01
4,aktobe,Aktobe Region,959184,2026-10-01


## 6 · Match current OSM ADM1 boundaries to the 20 official regions

Alias matching is explicit and auditable; the notebook prints unmatched boundary labels and **stops if 20/20 regions are not recognized**. Names may differ between OSM revisions. Do not silently assign `Almaty` (city) to `Almaty Region`. If the current OSM snapshot uses a new spelling, append a verified spelling to the alias dictionary and rerun.

In [13]:
REGION_ALIASES = {
 "astana": ["Astana", "Astana city", "Астана", "Астана қаласы", "город Астана", "Нур-Султан", "Nur-Sultan"],
 "almaty_city": ["Almaty", "Almaty city", "Алматы", "Алматы қаласы", "город Алматы"],
 "shymkent": ["Shymkent", "Shymkent city", "Шымкент", "Шымкент қаласы", "город Шымкент"],
 "almaty_region": ["Almaty Region", "Almaty Oblast", "Алматинская область", "Алматы облысы"],
 "aktobe": ["Aktobe Region", "Aktobe Oblast", "Актюбинская область", "Ақтөбе облысы", "Актобе область"],
 "atyrau": ["Atyrau Region", "Atyrau Oblast", "Атырауская область", "Атырау облысы"],
 "akmola": ["Akmola Region", "Akmola Oblast", "Акмолинская область", "Ақмола облысы"],
 "west_kazakhstan": ["West Kazakhstan Region", "West Kazakhstan Oblast", "Западно-Казахстанская область", "Батыс Қазақстан облысы", "Batys Qazaqstan oblysy"],
 "east_kazakhstan": ["East Kazakhstan Region", "East Kazakhstan Oblast", "Восточно-Казахстанская область", "Шығыс Қазақстан облысы", "Shyghys Qazaqstan oblysy"],
 "zhambyl": ["Zhambyl Region", "Zhambyl Oblast", "Жамбылская область", "Жамбыл облысы", "Джамбульская область"],
 "karaganda": ["Karaganda Region", "Karaganda Oblast", "Карагандинская область", "Қарағанды облысы", "Karagandy Region", "Qaragandy Region"],
 "kostanay": ["Kostanay Region", "Kostanay Oblast", "Костанайская область", "Қостанай облысы", "Qostanay Region"],
 "kyzylorda": ["Kyzylorda Region", "Kyzylorda Oblast", "Кызылординская область", "Қызылорда облысы", "Qyzylorda Region"],
 "mangystau": ["Mangystau Region", "Mangystau Oblast", "Мангистауская область", "Маңғыстау облысы", "Mangistau Region"],
 "north_kazakhstan": ["North Kazakhstan Region", "North Kazakhstan Oblast", "Северо-Казахстанская область", "Солтүстік Қазақстан облысы"],
 "pavlodar": ["Pavlodar Region", "Pavlodar Oblast", "Павлодарская область", "Павлодар облысы"],
 "turkistan": ["Turkistan Region", "Turkestan Region", "Turkistan Oblast", "Туркестанская область", "Түркістан облысы", "Оңтүстік Қазақстан облысы"],
 "abay": ["Abay Region", "Abai Region", "Abay Oblast", "Абай облысы", "область Абай"],
 "zhetisu": ["Zhetisu Region", "Jetisu Region", "Zhetysu Region", "Жетісу облысы", "область Жетісу"],
 "ulytau": ["Ulytau Region", "Ulytau Oblast", "Ұлытау облысы", "область Ұлытау"],
}

def norm_name(value):
    if not isinstance(value, str): return ""
    value = unicodedata.normalize("NFKC", value).casefold().replace("ё", "е")
    value = re.sub(r"[^\wа-яәіңғүұқөһі]+", " ", value, flags=re.UNICODE)
    return " ".join(value.split())

alias_lookup = {}
for code_key, variants in REGION_ALIASES.items():
    for variant in variants:
        norm = norm_name(variant)
        if norm in alias_lookup and alias_lookup[norm] != code_key:
            raise ValueError(f"Ambiguous region alias: {variant}")
        alias_lookup[norm] = code_key

def region_from_feature(feature):
    props = feature.get("properties") or {}
    if str(props.get("boundary", "")).lower() != "administrative": return None
    if str(props.get("admin_level", "")) != "4": return None
    if props.get("@type") != "relation": return None
    guesses = {alias_lookup[norm_name(props.get(key))] for key in ["name:en", "name", "name:ru", "name:kk"]
               if norm_name(props.get(key)) in alias_lookup}
    if len(guesses) != 1: return None
    geom = feature.get("geometry")
    if not geom: return None
    try:
        poly = shape(geom)
        if poly.is_empty or poly.geom_type not in {"Polygon", "MultiPolygon"}: return None
    except Exception: return None
    return {"region_code": next(iter(guesses)), "geometry": poly}

raw_boundaries = list(geojson_records(REGION_SEQ))
regions_records = [region_from_feature(f) for f in raw_boundaries]
regions_records = [r for r in regions_records if r is not None]
regions = gpd.GeoDataFrame(regions_records, geometry="geometry", crs="EPSG:4326")
if regions.empty:
    raise RuntimeError("No ADM1 boundaries matched: inspect OSM names and REGION_ALIASES")
regions = regions.dissolve(by="region_code", as_index=False)
matched = set(regions.region_code)
missing_region_codes = sorted(set(population.region_code) - matched)
print("ADM1 polygons recognized:", len(regions), "out of 20")
print("Unmatched official region codes:", missing_region_codes)
print("Other ADM1 labels (for debugging):")
for f in raw_boundaries:
    p = f.get("properties") or {}
    if p.get("boundary") == "administrative" and str(p.get("admin_level")) == "4" and region_from_feature(f) is None:
        print({k:p.get(k) for k in ("name","name:en","name:ru","name:kk","@id")})
if missing_region_codes:
    raise RuntimeError("Please map missing OSM admin names in REGION_ALIASES. Do NOT publish partial regional rankings.")
regions = regions.merge(population, on="region_code", validate="one_to_one")
regions["area_km2"] = regions.to_crs("EPSG:6933").area / 1e6
regions.to_file(REGION_GJ, driver="GeoJSON")
regions.drop(columns="geometry").to_csv(REGION_CSV, index=False)
print("All 20 administrative regions matched; polygons stored:", REGION_GJ)

ADM1 polygons recognized: 20 out of 20
Unmatched official region codes: []
Other ADM1 labels (for debugging):
All 20 administrative regions matched; polygons stored: /content/kazakhstan_poi_47/outputs/regions.geojson


## 7 · Spatial join in chunks + data quality audit

`GeoPandas.sjoin(..., predicate='within')` assigns POIs to polygons by coordinates. Counts outside an ADM1 polygon are kept as `NULL` region and reported (rather than relocated to the nearest province). Any multiple matches are counted and resolved deterministically; inspect overlaps in a full run. Parquet written in bounded Arrow batches.

In [14]:
ENRICH_SCHEMA = RAW_SCHEMA.append(pa.field("region_code", pa.string()))

def spatial_join_parquet(source, destination, region_polys, batch_rows=20000):
    polygons = region_polys[["region_code", "geometry"]].copy()
    writer = pq.ParquetWriter(destination, ENRICH_SCHEMA, compression="zstd")
    stats = {"rows": 0, "outside_boundaries": 0, "ambiguous_assignments": 0}
    try:
        reader = pq.ParquetFile(source)
        for batch in reader.iter_batches(batch_size=batch_rows):
            frame = batch.to_pandas()
            points = gpd.GeoDataFrame(frame, geometry=gpd.points_from_xy(frame.longitude, frame.latitude), crs="EPSG:4326")
            points["__rowid"] = np.arange(len(points))
            joined = gpd.sjoin(points, polygons, how="left", predicate="within")
            stats["ambiguous_assignments"] += int(joined.duplicated(subset="__rowid").sum())
            assigned = (joined.sort_values(["__rowid", "region_code"], na_position="last")
                        .drop_duplicates("__rowid").sort_values("__rowid"))
            out = frame.copy()
            out["region_code"] = assigned.region_code.to_numpy()
            stats["rows"] += len(out)
            stats["outside_boundaries"] += int(out.region_code.isna().sum())
            writer.write_table(pa.Table.from_pandas(out, schema=ENRICH_SCHEMA, preserve_index=False))
    finally:
        writer.close()
    return stats

if not ENRICHED_PQ.exists():
    SPATIAL_STATS = spatial_join_parquet(RAW_PQ, ENRICHED_PQ, regions, CHUNK_ROWS)
else:
    SPATIAL_STATS = {"rows": pq.read_metadata(ENRICHED_PQ).num_rows, "cached": True}
print(SPATIAL_STATS)
print("Parquet sizes (MiB):", {p.name:round(p.stat().st_size/1024**2,1) for p in (RAW_PQ, ENRICHED_PQ)})

{'rows': 126778, 'outside_boundaries': 1243, 'ambiguous_assignments': 0}
Parquet sizes (MiB): {'poi_raw.parquet': 4.0, 'poi_enriched.parquet': 4.0}


In [15]:
poi = pd.read_parquet(ENRICHED_PQ)
assert not poi.empty and poi.osm_key.is_unique, "Duplicate/missing OSM entity keys"
assert set(poi.region_code.dropna()).issubset(set(regions.region_code))
print("Rows:", f"{len(poi):,}")
print("Missing fraction:\n", poi.isna().mean().sort_values(ascending=False).to_string())
print("Geometry types:\n", poi.geometry_type.value_counts().to_string())
print("Category distribution:\n", poi.category.value_counts().to_string())
print("Raw frame memory MiB:", round(poi.memory_usage(deep=True).sum() / 2**20, 2))
optimized = poi.copy()
for col in ["osm_type", "main_tag", "subcategory", "category", "geometry_type", "region_code"]:
    optimized[col] = optimized[col].astype("category")
print("Optimized categorical memory MiB:", round(optimized.memory_usage(deep=True).sum()/2**20, 2))
qa = {"rows":len(poi), "unique_osm_keys":int(poi.osm_key.nunique()),
      "missing_region_count":int(poi.region_code.isna().sum()),
      "missing_name_count":int(poi.name.isna().sum()),
      "invalid_coordinates":int((~poi.longitude.between(46,88) | ~poi.latitude.between(40,56)).sum()),
      "duplicate_osm_keys":int(poi.osm_key.duplicated().sum()),
      "category_counts":{str(k):int(v) for k,v in poi.category.value_counts().items()},
      "baseline_memory_mib":round(poi.memory_usage(deep=True).sum()/2**20,2),
      "optimized_memory_mib":round(optimized.memory_usage(deep=True).sum()/2**20,2)}
(OUT/"quality_report.json").write_text(json.dumps(qa, indent=2, ensure_ascii=False), encoding="utf-8")
print("QA saved:", OUT/"quality_report.json")

Rows: 126,778
Missing fraction:
 name             0.555546
region_code      0.009805
osm_key          0.000000
osm_id           0.000000
osm_type         0.000000
subcategory      0.000000
main_tag         0.000000
category         0.000000
latitude         0.000000
longitude        0.000000
geometry_type    0.000000
Geometry types:
 geometry_type
Point           72247
MultiPolygon    51990
LineString       2541
Category distribution:
 category
Retail             31767
Other amenities    20424
Recreation         17139
Transport          16582
Food & drink        9383
Tourism             6968
Public services     6701
Education           6664
Healthcare          5711
Finance             5439
Raw frame memory MiB: 57.44
Optimized categorical memory MiB: 17.37
QA saved: /content/kazakhstan_poi_47/outputs/quality_report.json


## 8 · Regional and city POI analytics

Population denominators match *first-level administrative areas* only. `Astana city`, `Almaty city`, and `Shymkent city` are comparable city-regions. Other first-level oblasts contain rural areas and **cannot** be called individual cities; show them as *regions*. Density per 100 km² uses projected polygon area, not degree².

In [16]:
region_category = (poi.dropna(subset=["region_code"])
                   .groupby(["region_code", "category"], observed=True)
                   .size().rename("poi_count").reset_index())
region_totals = poi.dropna(subset=["region_code"]).groupby("region_code").size().rename("poi_count").reset_index()
region_stats = (regions.drop(columns="geometry")
                .merge(region_totals, on="region_code", how="left")
                .fillna({"poi_count":0}))
region_stats["poi_count"] = region_stats.poi_count.astype(int)
region_stats["per_100k"] = region_stats.poi_count / region_stats.population * 100_000
region_stats["per_100km2"] = region_stats.poi_count / region_stats.area_km2 * 100
region_stats = region_stats.sort_values("per_100k", ascending=False)
# Robust IQR *flag*, not a deletion rule; variation between regions is substantive.
q1, q3 = region_stats.per_100k.quantile([0.25, 0.75])
iqr = q3 - q1
region_stats["density_iqr_outlier"] = (region_stats.per_100k < q1-1.5*iqr) | (region_stats.per_100k > q3+1.5*iqr)
region_stats.to_csv(OUT/"region_stats.csv", index=False)
region_stats.loc[region_stats.density_iqr_outlier].to_csv(OUT/"regional_outliers_iqr.csv", index=False)
print("IQR-flagged regions (not deleted):", region_stats.loc[region_stats.density_iqr_outlier, "region_name"].tolist())
region_category.to_csv(OUT/"region_category.csv", index=False)
print(region_stats[["region_name","poi_count","per_100k","per_100km2"]].to_string(index=False))
CITY_CODES = ["astana", "almaty_city", "shymkent"]
city_stats = region_stats[region_stats.region_code.isin(CITY_CODES)].copy()
city_stats.to_csv(OUT/"city_stats.csv", index=False)

IQR-flagged regions (not deleted): []
            region_name  poi_count   per_100k  per_100km2
            Almaty city      23460 980.726180 3333.162167
 East Kazakhstan Region       6402 895.433457    6.588930
          Aktobe Region       8432 879.080552    2.802363
          Akmola Region       6495 815.703709    4.448373
 West Kazakhstan Region       5535 797.231080    3.657623
          Atyrau Region       5560 773.749302    4.335242
         Zhetisu Region       5074 743.813768    4.311464
        Kostanay Region       5621 687.073100    2.867095
North Kazakhstan Region       3214 629.972500    3.278022
          Almaty Region      10179 628.945973    9.712698
       Karaganda Region       6860 606.899187    2.875116
            Abay Region       3557 602.717907    1.916376
        Pavlodar Region       4323 582.926444    3.474429
            Astana city       9411 551.269660 1175.948844
       Kyzylorda Region       3619 428.698517    1.595368
          Ulytau Region        807

## 9 · Visualizations

Exports HTML figures for the app/report. All plotted numbers are computed from your extract; no synthetic conclusions. The map is intentionally capped at 2,500 POIs for responsiveness and is labelled as a sample.

In [17]:
fig1 = px.bar(region_stats, x="region_name", y="poi_count", title="Mapped POIs by first-level administrative region")
fig1.update_layout(xaxis_tickangle=-55)
fig1.write_html(FIG/"01_region_counts.html", include_plotlyjs="cdn")
fig1.show()

fig2 = px.bar(region_stats, x="region_name", y="per_100k", title="Mapped POIs per 100,000 residents (2026-10-01)")
fig2.update_layout(xaxis_tickangle=-55)
fig2.write_html(FIG/"02_region_per_100k.html", include_plotlyjs="cdn")
fig2.show()

In [18]:
city_distribution = region_category[region_category.region_code.isin(CITY_CODES)].merge(
    regions[["region_code","region_name"]], on="region_code", how="left")
fig3 = px.bar(city_distribution, x="region_name", y="poi_count", color="category", barmode="stack",
              title="Astana vs Almaty vs Shymkent: mapped POI mix")
fig3.write_html(FIG/"03_city_mix.html", include_plotlyjs="cdn")
fig3.show()

heat = (region_category.pivot(index="region_code", columns="category", values="poi_count")
        .reindex(regions.region_code).fillna(0))
fig4 = px.imshow(heat, aspect="auto", labels={"x":"Category", "y":"Region code", "color":"Mapped POIs"},
                 title="POI count heatmap by region and type")
fig4.write_html(FIG/"04_category_heatmap.html", include_plotlyjs="cdn")
fig4.show()

In [19]:
# Stable sampling for a responsive, interactive Plotly map. This is NOT a complete POI map.
map_sample = poi.dropna(subset=["region_code"]).sample(
    n=min(2500, int(poi.region_code.notna().sum())), random_state=RANDOM_STATE)
def scatter_osm_map(data, **kwargs):
    if hasattr(px, "scatter_map"):
        return px.scatter_map(data, map_style="open-street-map", **kwargs)
    return px.scatter_mapbox(data, mapbox_style="open-street-map", **kwargs)

fig5 = scatter_osm_map(map_sample, lat="latitude", lon="longitude", color="category",
                      hover_name="name", hover_data=["region_code","subcategory"],
                      center={"lat":48.0, "lon":67.0}, zoom=3,
                      title=f"Interactive Kazakhstan POI map — sampled {len(map_sample):,} of {len(poi):,} entities")
fig5.update_layout(height=620)
fig5.write_html(FIG/"05_poi_map_sample.html", include_plotlyjs="cdn")
fig5.show()

## 10 · Two pre-specified hypotheses

**H1 (category composition):** across Astana, Almaty and Shymkent, category distributions are identical. We test the *entire contingency table* using a chi-squared test; report Cramér's V effect size, not only the p-value.

**H2 (local concentration):** the distributions of *nearest same-category POI distances* for healthcare POIs are equal between Astana and Almaty. We calculate nearest-neighbor distances in kilometers with a haversine BallTree and compare with Mann–Whitney U.

**Caveat:** OSM is an observed, edited, spatially dependent point pattern rather than an IID sample. Inferential p-values are **descriptive exploratory signals**, not causal or nationally representative results. The 2017/2026 boundary mismatch is explicitly avoided.

In [20]:
THRESHOLD = 0.05
city_pois = poi[poi.region_code.isin(CITY_CODES)].copy()
contingency = pd.crosstab(city_pois.region_code, city_pois.category)
if contingency.shape[0] < 3 or contingency.shape[1] < 2:
    raise RuntimeError("Insufficient city/category coverage for H1")
chi2, p_chi, dof, expected = chi2_contingency(contingency)
n = contingency.to_numpy().sum()
cramers_v = float(np.sqrt(chi2 / (n * (min(contingency.shape) - 1))))
h1 = {"hypothesis":"H1 city category distributions equal", "method":"chi_square",
      "chi2":float(chi2), "degrees_of_freedom":int(dof), "p_value":float(p_chi),
      "cramers_v":cramers_v, "min_expected_cell":float(expected.min()),
      "reject_at_0_05": bool(p_chi < THRESHOLD)}
print(h1)
contingency.to_csv(OUT/"hypothesis1_contingency.csv")

{'hypothesis': 'H1 city category distributions equal', 'method': 'chi_square', 'chi2': 1163.7958111478324, 'degrees_of_freedom': 18, 'p_value': 6.370977182342534e-236, 'cramers_v': 0.12691339079390582, 'min_expected_cell': 39.56553270407175, 'reject_at_0_05': True}


In [21]:
EARTH_RADIUS_KM = 6371.0088

def nearest_neighbor_km(frame):
    if len(frame) < 3: return np.array([])
    coords = np.radians(frame[["latitude", "longitude"]].to_numpy(dtype=float))
    tree = BallTree(coords, metric="haversine")
    dists, indexes = tree.query(coords, k=2)
    return dists[:,1] * EARTH_RADIUS_KM

health = poi[poi.category == "Healthcare"]
d_asta = nearest_neighbor_km(health[health.region_code == "astana"])
d_alma = nearest_neighbor_km(health[health.region_code == "almaty_city"])
if min(len(d_asta), len(d_alma)) >= 10:
    u, p_mwu = mannwhitneyu(d_asta, d_alma, alternative="two-sided")
    h2 = {"hypothesis":"H2 healthcare nearest-neighbor distance distributions equal",
          "method":"Mann-Whitney U", "n_astana":int(len(d_asta)), "n_almaty":int(len(d_alma)),
          "median_astana_km":float(np.median(d_asta)), "median_almaty_km":float(np.median(d_alma)),
          "u":float(u), "p_value":float(p_mwu), "reject_at_0_05":bool(p_mwu < THRESHOLD)}
else:
    h2 = {"hypothesis":"H2", "status":"not_tested", "reason":"Need >=10 healthcare POIs per city"}
print(h2)
(OUT/"hypotheses.json").write_text(json.dumps([h1,h2], indent=2, ensure_ascii=False), encoding="utf-8")

fig6 = px.box(pd.DataFrame({"city": ["Astana"]*len(d_asta) + ["Almaty"]*len(d_alma),
                            "nearest_healthcare_km": np.r_[d_asta, d_alma]}),
              x="city", y="nearest_healthcare_km", points=False,
              title="Nearest other healthcare POI distance (mapped features)")
fig6.write_html(FIG/"06_healthcare_nearest_neighbor.html", include_plotlyjs="cdn")
fig6.show()

{'hypothesis': 'H2 healthcare nearest-neighbor distance distributions equal', 'method': 'Mann-Whitney U', 'n_astana': 408, 'n_almaty': 839, 'median_astana_km': 0.12688172842842144, 'median_almaty_km': 0.10343106208061603, 'u': 188605.0, 'p_value': 0.003451668531607622, 'reject_at_0_05': True}


## 11 · Spatial clustering using haversine DBSCAN

Clustering is shown for each of the three city-regions, using **500 metres (`eps=0.5 km`) and `min_samples=8`**. Distances use lat/lon in radians and the haversine metric: coordinates are **not** treated as Euclidean degrees. Sampling takes place only for city extracts above `MAX_CLUSTER_POINTS`; if sampled, results are **exploratory and not valid as the city's full cluster count**.

In [22]:
CLUSTER_EPS_KM = 0.5
CLUSTER_MIN_SAMPLES = 8

def compute_city_clusters(data, city_code, sample_max=12000):
    city = data[data.region_code == city_code].copy()
    was_sampled = len(city) > sample_max
    if was_sampled: city = city.sample(n=sample_max, random_state=RANDOM_STATE).copy()
    if city.empty: return city.assign(cluster_id=pd.Series(dtype="int64")), {"n":0, "sampled":was_sampled}
    coords = np.radians(city[["latitude", "longitude"]].to_numpy(dtype=float))
    model = DBSCAN(eps=CLUSTER_EPS_KM/EARTH_RADIUS_KM,
                   min_samples=CLUSTER_MIN_SAMPLES, metric="haversine", algorithm="ball_tree")
    city["cluster_id"] = model.fit_predict(coords).astype(int)
    return city, {"n":int(len(city)), "sampled":bool(was_sampled),
                  "clusters":int(len(set(city.cluster_id) - {-1})),
                  "noise_fraction":float((city.cluster_id == -1).mean())}

cluster_tables = []
cluster_report = {}
for city in CITY_CODES:
    assigned, details = compute_city_clusters(poi, city, MAX_CLUSTER_POINTS)
    assigned["cluster_city"] = city
    cluster_tables.append(assigned[["osm_key", "cluster_city", "cluster_id", "latitude", "longitude", "category"]])
    cluster_report[city] = details
clusters = pd.concat(cluster_tables, ignore_index=True)
clusters.to_parquet(OUT/"clusters.parquet", index=False)
(OUT/"clustering_report.json").write_text(json.dumps(cluster_report, indent=2), encoding="utf-8")
print(json.dumps(cluster_report, indent=2))

# Representative point map of Astana clusters; plot a bounded sample for readability.
astana_clusters = clusters[clusters.cluster_city=="astana"].copy()
if not astana_clusters.empty:
    astana_clusters["cluster_label"] = astana_clusters.cluster_id.astype(str)
    fig7 = scatter_osm_map(astana_clusters, lat="latitude", lon="longitude", color="cluster_label",
                          center={"lat":51.15, "lon":71.43}, zoom=10,
                          title="Astana DBSCAN clusters (noise = -1; sampled when needed)")
    fig7.update_layout(height=650)
    fig7.write_html(FIG/"07_astana_clusters.html", include_plotlyjs="cdn")
    fig7.show()

{
  "astana": {
    "n": 9411,
    "sampled": false,
    "clusters": 25,
    "noise_fraction": 0.02082669216873871
  },
  "almaty_city": {
    "n": 12000,
    "sampled": true,
    "clusters": 45,
    "noise_fraction": 0.043416666666666666
  },
  "shymkent": {
    "n": 3256,
    "sampled": false,
    "clusters": 20,
    "noise_fraction": 0.10933660933660934
  }
}


## 12 · SQLite database + ER model

Logical ER relationships:

```mermaid
erDiagram
    REGION ||--o{ POI : contains
    CATEGORY ||--o{ POI : classifies
    REGION {
      TEXT region_code PK
      TEXT region_name
      INTEGER population
      REAL area_km2
    }
    CATEGORY {
      TEXT category_name PK
    }
    POI {
      TEXT osm_key PK
      TEXT region_code FK
      TEXT category_name FK
      REAL latitude
      REAL longitude
    }
```

A nullable `POI.region_code` retains unmatched coordinates for QA. SQL indexes are created on region/category and location.

In [23]:
if DB_PATH.exists(): DB_PATH.unlink()  # build reproducibly from the cleaned Parquet
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA foreign_keys=ON")
conn.executescript("""
CREATE TABLE region (
  region_code TEXT PRIMARY KEY, region_name TEXT NOT NULL UNIQUE,
  population INTEGER NOT NULL CHECK (population > 0),
  population_date TEXT NOT NULL, area_km2 REAL NOT NULL CHECK (area_km2 > 0)
);
CREATE TABLE category (category_name TEXT PRIMARY KEY);
CREATE TABLE poi (
  osm_key TEXT PRIMARY KEY, osm_type TEXT NOT NULL, osm_id INTEGER NOT NULL,
  name TEXT, main_tag TEXT NOT NULL, subcategory TEXT NOT NULL,
  category_name TEXT NOT NULL REFERENCES category(category_name),
  region_code TEXT REFERENCES region(region_code),
  latitude REAL NOT NULL, longitude REAL NOT NULL, geometry_type TEXT NOT NULL
);
CREATE INDEX idx_poi_region_category ON poi(region_code, category_name);
CREATE INDEX idx_poi_category ON poi(category_name);
""")
conn.executemany("INSERT INTO region VALUES (?, ?, ?, ?, ?)",
    regions[["region_code","region_name","population","population_date","area_km2"]].itertuples(index=False, name=None))
conn.executemany("INSERT INTO category VALUES (?)", [(name,) for name in sorted(poi.category.unique())])
INSERT = "INSERT INTO poi VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)"
DB_COLUMNS = ["osm_key","osm_type","osm_id","name","main_tag","subcategory","category",
              "region_code","latitude","longitude","geometry_type"]
for batch in pq.ParquetFile(ENRICHED_PQ).iter_batches(batch_size=CHUNK_ROWS):
    df = batch.to_pandas()[DB_COLUMNS].astype(object)
    df = df.where(pd.notna(df), None)
    conn.executemany(INSERT, df.itertuples(index=False, name=None))
conn.commit()
assert conn.execute("SELECT COUNT(*) FROM poi").fetchone()[0] == len(poi)
print("SQLite rows:", conn.execute("SELECT COUNT(*) FROM poi").fetchone()[0], "db:", DB_PATH)

SQLite rows: 126778 db: /content/kazakhstan_poi_47/outputs/kazakhstan_poi.sqlite


## 13 · Eight SQL analytical queries (includes JOIN, GROUP BY/HAVING, CTE, window functions)

Each query is executed **against SQLite** and its result exported to CSV for downstream app/report use. The code below also provides the raw SQL strings required for explanation during defense.

In [24]:
SQL_QUERIES = {
"01_region_poi_counts": """SELECT r.region_name, COUNT(p.osm_key) AS poi_count
FROM region r LEFT JOIN poi p ON p.region_code=r.region_code
GROUP BY r.region_code ORDER BY poi_count DESC;""",
"02_poi_per_100k": """SELECT r.region_name, r.population,
COUNT(p.osm_key) AS poi_count,
ROUND(100000.0*COUNT(p.osm_key)/r.population, 2) AS poi_per_100k
FROM region r LEFT JOIN poi p ON r.region_code=p.region_code
GROUP BY r.region_code ORDER BY poi_per_100k DESC;""",
"03_city_category_counts": """SELECT r.region_name, c.category_name, COUNT(p.osm_key) AS poi_count
FROM poi p JOIN region r ON p.region_code=r.region_code
JOIN category c ON p.category_name=c.category_name
WHERE r.region_code IN ('astana','almaty_city','shymkent')
GROUP BY r.region_code, c.category_name ORDER BY r.region_name,poi_count DESC;""",
"04_categories_above_threshold": """SELECT category_name, COUNT(*) AS poi_count
FROM poi GROUP BY category_name HAVING COUNT(*) >= 100
ORDER BY poi_count DESC;""",
"05_coverage_cte": """WITH category_counts AS (
 SELECT region_code, COUNT(DISTINCT category_name) AS categories_observed, COUNT(*) AS poi_count
 FROM poi WHERE region_code IS NOT NULL GROUP BY region_code)
SELECT r.region_name, COALESCE(c.categories_observed,0) AS categories_observed,
COALESCE(c.poi_count,0) AS poi_count
FROM region r LEFT JOIN category_counts c USING(region_code)
ORDER BY categories_observed DESC, poi_count DESC;""",
"06_region_category_ranking": """WITH counts AS (
 SELECT region_code, category_name, COUNT(*) AS poi_count
 FROM poi WHERE region_code IS NOT NULL GROUP BY region_code,category_name)
SELECT r.region_name, c.category_name, c.poi_count,
DENSE_RANK() OVER (PARTITION BY c.region_code ORDER BY c.poi_count DESC) AS category_rank
FROM counts c JOIN region r ON c.region_code=r.region_code
ORDER BY r.region_name, category_rank;""",
"07_region_vs_national_share": """WITH counts AS (
 SELECT region_code,category_name,COUNT(*) AS count_in_region
 FROM poi WHERE region_code IS NOT NULL GROUP BY region_code,category_name),
national AS (SELECT category_name,SUM(count_in_region) AS count_all FROM counts GROUP BY category_name)
SELECT r.region_name,c.category_name,c.count_in_region,n.count_all,
ROUND(100.0*c.count_in_region/NULLIF(n.count_all,0),2) AS share_of_national_category_pct
FROM counts c JOIN national n USING(category_name) JOIN region r USING(region_code)
ORDER BY share_of_national_category_pct DESC;""",
"08_missing_names_or_regions": """SELECT category_name,COUNT(*) AS total,
SUM(CASE WHEN name IS NULL OR TRIM(name)='' THEN 1 ELSE 0 END) AS missing_name,
SUM(CASE WHEN region_code IS NULL THEN 1 ELSE 0 END) AS missing_region
FROM poi GROUP BY category_name ORDER BY total DESC;""",
}
SQL_OUTPUT = {}
for key, query in SQL_QUERIES.items():
    result = pd.read_sql_query(query, conn)
    result.to_csv(QUERY_DIR / f"{key}.csv", index=False)
    (QUERY_DIR / f"{key}.sql").write_text(query, encoding="utf-8")
    SQL_OUTPUT[key] = len(result)
print("Executed and exported", len(SQL_QUERIES), "SQL queries:", SQL_OUTPUT)
assert len(SQL_QUERIES) >= 8

Executed and exported 8 SQL queries: {'01_region_poi_counts': 20, '02_poi_per_100k': 20, '03_city_category_counts': 30, '04_categories_above_threshold': 10, '05_coverage_cte': 20, '06_region_category_ranking': 200, '07_region_vs_national_share': 200, '08_missing_names_or_regions': 10}


## 14 · Baseline vs Pandas vs Polars: time + peak RAM

**Identical task:** count POIs by `(region_code, category)` for *assigned* POIs. **Baseline** = SQLite SQL. Two extra approaches = Pandas and Polars. All must return exactly the same aggregate counts. To avoid measuring a long-lived notebook's existing memory, the benchmark runs each candidate in a **fresh Python child process** and samples its peak RSS, including imports. Differences reflect both file format and execution engine; they are not isolated engine-only speedups.

In [25]:
BENCHMARK_CODE = r"""
import json,sqlite3,sys
mode,parquet,db=sys.argv[1:]
if mode=="pandas":
 import pandas as pd
 a=pd.read_parquet(parquet,columns=["region_code","category"])
 a=a.dropna(subset=["region_code"])
 res=a.groupby(["region_code","category"]).size().reset_index(name="n")
 rows=[(str(a),str(b),int(n)) for a,b,n in res.itertuples(index=False,name=None)]
elif mode=="polars":
 import polars as pl
 a=(pl.scan_parquet(parquet).select(["region_code","category"])
    .filter(pl.col("region_code").is_not_null())
    .group_by(["region_code","category"]).len().collect())
 rows=[(str(a),str(b),int(n)) for a,b,n in a.iter_rows()]
elif mode=="sqlite":
 conn=sqlite3.connect(db)
 rows=conn.execute("SELECT region_code,category_name,COUNT(*) FROM poi WHERE region_code IS NOT NULL GROUP BY region_code,category_name").fetchall()
else: raise ValueError(mode)
print(json.dumps(sorted(rows),separators=(",",":")))
"""
import psutil

def benchmark_mode(mode):
    cmd = [sys.executable, "-c", BENCHMARK_CODE, mode, str(ENRICHED_PQ), str(DB_PATH)]
    start = time.perf_counter()
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    proc = psutil.Process(process.pid)
    peak_rss = 0
    while process.poll() is None:
        try:
            peak_rss = max(peak_rss, proc.memory_info().rss)
        except psutil.NoSuchProcess: break
        time.sleep(0.02)
    stdout, stderr = process.communicate()
    elapsed = time.perf_counter() - start
    if process.returncode:
        raise RuntimeError(f"Benchmark {mode} failed: {stderr}")
    return {"approach":mode, "time_seconds":round(elapsed,3),
            "peak_process_rss_mib":round(peak_rss/2**20,1), "result":json.loads(stdout)}

BENCH = [benchmark_mode(m) for m in ["sqlite","pandas","polars"]]
reference = BENCH[0]["result"]
assert all(record["result"] == reference for record in BENCH), "Aggregate mismatch across benchmark methods"
bench_out = pd.DataFrame([{k:v for k,v in record.items() if k != "result"} for record in BENCH])
bench_out["matches_baseline"] = True
bench_out.to_csv(OUT/"benchmark.csv", index=False)
print(bench_out.to_string(index=False))
conn.close()

approach  time_seconds  peak_process_rss_mib  matches_baseline
  sqlite         0.124                  17.6              True
  pandas         0.942                 131.1              True
  polars         0.449                  74.4              True


## 15 · Export manifest and Codex data contracts

The eventual Streamlit app **must read preprocessed** `outputs/*.parquet`, `*.csv`, `regions.geojson`, and SQLite; it must not download and reparse a 200+ MB PBF on every page load. Do not commit `data/raw/`, PBF, or oversized derived artifacts. Link to their source or publish a separate release where permitted under ODbL.

In [26]:
def file_summary(path):
    path = Path(path)
    return {"filename":path.name, "bytes":path.stat().st_size} if path.exists() else None

manifest = {
 "project":"Topic 47 — Kazakhstan OSM POI Analytics",
 "generated_utc":datetime.now(timezone.utc).isoformat(),
 "pbf_source":PBF_URL,
 "pbf_local_file_mtime_utc":PBF_LOCAL_FILE_MTIME_UTC,
 "pbf_size_bytes":PBF.stat().st_size,
 "population_source":POPULATION_SOURCE,
 "population_date":POPULATION_DATE,
 "population_total":int(population.population.sum()),
 "osm_attribution":"© OpenStreetMap contributors, ODbL 1.0; extract by Geofabrik",
 "poi_definition":"One record per OSM node/way/relation, priority amenity > shop > tourism > leisure",
 "regions_definition":"20 current OSM administrative level 4 polygon relations matched explicitly to BNS region codes",
 "spatial_method":"representative point for POI geometries; gpd.sjoin within to ADM1 polygons",
 "trainable_ml_extension": "GPU PyTorch autoencoder on 2km urban category-count grids; spatial-block split; PCA and mean baselines",
 "cluster_parameters":{"eps_km":CLUSTER_EPS_KM,"min_samples":CLUSTER_MIN_SAMPLES,"max_points_per_city":MAX_CLUSTER_POINTS},
 "files":{key:file_summary(path) for key,path in {
  "poi_enriched":ENRICHED_PQ,"regions":REGION_GJ,"region_stats":OUT/"region_stats.csv",
  "region_category":OUT/"region_category.csv","city_stats":OUT/"city_stats.csv",
  "sqlite":DB_PATH,"clusters":OUT/"clusters.parquet", "quality_report":OUT/"quality_report.json",
  "benchmark":OUT/"benchmark.csv","hypotheses":OUT/"hypotheses.json",
  "gpu_autoencoder": OUT/"urban_autoencoder.pt", "gpu_model_metrics": OUT/"gpu_model_metrics.csv",
  "gpu_model_report": OUT/"gpu_model_report.json",
  "urban_grid_profiles": OUT/"urban_grid_profiles.parquet"}.items()},
 "limitations":[
  "OSM mapping coverage unequal across regions; counts do not equal real service inventories",
  "POI polygons and lines reduced to representative points",
  "Population denominators are first-level region totals; only 3 standalone city-region comparisons",
  "DBSCAN may use sampled POIs for very large cities",
  "Snapshot mismatch: BNS 2026-10-01 vs latest Geofabrik PBF download date",
  "OSM node/area tagging may introduce distinct records for the same real-world establishment"
 ]
}
(OUT/"manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Manifest:", OUT/"manifest.json")
print("Top-level output files:", [p.name for p in sorted(OUT.iterdir()) if p.is_file()])

Manifest: /content/kazakhstan_poi_47/outputs/manifest.json
Top-level output files: ['benchmark.csv', 'city_stats.csv', 'clustering_report.json', 'clusters.parquet', 'gpu_model_metrics.csv', 'gpu_model_report.json', 'hypotheses.json', 'hypothesis1_contingency.csv', 'kazakhstan_poi.sqlite', 'manifest.json', 'pca_baseline.joblib', 'poi_enriched.parquet', 'poi_raw.parquet', 'population_2026_10_01.csv', 'quality_report.json', 'region_category.csv', 'region_stats.csv', 'regional_outliers_iqr.csv', 'regions.csv', 'regions.geojson', 'urban_autoencoder.pt', 'urban_autoencoder_scaler.joblib', 'urban_grid_profiles.parquet', 'urban_profile_kmeans.joblib']


## 16 · Export app data bundle (optional)

In Colab, use this final cell to bundle *processed* outputs for Codex. The raw `.osm.pbf` and GeoJSONSeq intermediates are not included. The archive can still be large; a local project checkout can use `outputs/` directly.

In [27]:
import shutil as _shutil
BUNDLE_ARCHIVE = _shutil.make_archive(str(ROOT / "topic47_app_data"), "zip", root_dir=ROOT, base_dir="outputs")
print("Created bundle:", BUNDLE_ARCHIVE, "size MiB:", round(Path(BUNDLE_ARCHIVE).stat().st_size/2**20, 1))
# Uncomment in Google Colab to download to your computer:
# from google.colab import files
# files.download(BUNDLE_ARCHIVE)

Created bundle: /content/kazakhstan_poi_47/topic47_app_data.zip size MiB: 16.9


## 17 · Reproducibility checks and publication guidance

Before passing to Codex: check (i) 20/20 region matches, (ii) valid population total 20,633,529, (iii) POI ID uniqueness and output counts, (iv) benchmark equality, (v) SQL result exports, (vi) maps loaded, and (vii) `manifest.json` and sources in the report.

For GitHub, commit this notebook, source modules produced by Codex, documentation, small aggregate CSVs, SQL query files, and screenshots; exclude `data/raw`, `*.osm.pbf`, `*.geojsonseq`, large Parquet and SQLite files unless deliberately distributed and compliant with the OSM database license. Include attribution **© OpenStreetMap contributors / ODbL 1.0** on maps, README, report, and data-download instructions.

**Potential improvement:** user-selectable extra cities (Karaganda, Aktobe, etc.) require reliable city municipal polygons or a carefully labelled radius-based approximation. Do not treat every *oblast* as its capital city.

## 18 · Download all processed outputs for Codex CLI

**Important:** `.ipynb` files store code, execution logs, and charts, **not** the trained `.pt` checkpoint, SQLite database, or processed Parquet files. Those files currently live in the Colab session. Run the following cell **in the same Colab runtime after the notebook finishes** to download the app-ready outputs bundle (`topic47_app_data.zip`). If the Colab runtime was reset, first rerun the analysis.

The previous run successfully created a **16.9 MiB** archive but did not trigger a browser download. This cell does that automatically on Colab.

In [28]:
from pathlib import Path
import shutil

# The complete set of artifacts produced by the nationwide analysis.
project_root = Path.cwd() / "kazakhstan_poi_47"
archive = project_root / "topic47_app_data.zip"
required = [
    project_root / "outputs" / "urban_autoencoder.pt",
    project_root / "outputs" / "urban_grid_profiles.parquet",
    project_root / "outputs" / "poi_enriched.parquet",
    project_root / "outputs" / "kazakhstan_poi.sqlite",
    project_root / "outputs" / "manifest.json",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Analysis outputs are missing from this Colab session; rerun the previous cells: "
        + ", ".join(missing)
    )

# Rebuild the archive so it includes the most recent analysis outputs.
archive = Path(shutil.make_archive(
    str(archive.with_suffix("")), "zip", root_dir=project_root, base_dir="outputs"
))
print(f"Ready: {archive} ({archive.stat().st_size / 2**20:.1f} MiB)")
try:
    from google.colab import files
except ImportError:
    print("Not running inside Colab; copy the ZIP from the path above.")
else:
    files.download(str(archive))


Ready: /content/kazakhstan_poi_47/topic47_app_data.zip (16.9 MiB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Interpretation notes for the report and Codex

- **Model:** GPU-trained PyTorch autoencoder is genuinely fitted on OSM-derived grid/category counts. It is **unsupervised reconstruction**, not supervised prediction and not an image model. The test split uses disjoint 10 km spatial blocks, but nearby blocks can still be correlated.
- **Evaluation:** MSE/MAE are calculated on *standardized log1p(category-count)* features, so they must not be described as an error in actual numbers of amenities. The PCA comparison uses the same 4-dimensional latent size.
- **Grid:** The `EPSG:6933` 2,000-unit square grid is in an equal-area projected CRS. Each cell has approximately **4 km² projected area**; the cells do **not** correspond to 2 km-by-2 km ground-distance squares everywhere.
- **Spatial inference:** the chi-square and Mann–Whitney p-values are *exploratory* because neighboring amenities are spatially dependent, violating simple independence assumptions. For stronger statistical inference, use spatial block permutation/bootstrap.
- **DBSCAN:** the Almaty city analysis randomly sampled 12,000 of 23,460 mapped POIs. Its 45 clusters are **sample-based**, not a definitive cluster count for all Almaty POIs.
- **Coverage:** 1,243 POIs (~0.98%) were not assigned an administrative polygon. OSM completeness varies across places; POI counts do not measure the true stock of businesses/services.
- **City definitions:** the comparison covers the three independent city-regions (Astana, Almaty, Shymkent), while the other 17 records are oblasts, not cities.
- **Data exports:** `urban_autoencoder.pt`, the scaler and KMeans joblib files, `urban_grid_profiles.parquet`, `poi_enriched.parquet`, `regions.geojson`, regional CSVs, SQL result files and SQLite are required for the Codex handoff. They must be downloaded via the ZIP above.
